# Introduction (updated from original)

This notebook takes the conserved network produced in NetColoc and generates a hierarchical systems map of protein communities. The systems map is annotated using GO enrichment to identify conserved functions underlying compulsion/psychiatric disorders in humans and dogs. We validate the role of protein communities by examining the phenotypic response in mice upon genetic perturbation.

**Additional Visualizations**  
* All visualizations of the network, hierarchy and communities can be found in the accompanying Cytoscape file.



# Set Up

In [ ]:
# load required libraries
import os
import sys
import ndex2
from getpass import getpass
import cdapsutil
import pandas as pd
import mygene
mg = mygene.MyGeneInfo()
from gprofiler import GProfiler
gp = GProfiler("MyToolName/0.1")

import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import fdrcorrection

In [2]:
# Set up directory paths. Defaults to the Data/ and Figures/ folders within the repository. 
# #i.e. /proj/snic2022-6-229/private/Darwin_Affy/NEW_DA_IMP_MERGED/2024-02-08/netcoloc/CrossSpeciesBMI
cwd = os.path.dirname(os.getcwd())
DATADIR = os.path.join(cwd, "Data/")
FIGDIR = os.path.join(cwd, "Figures/")

In [3]:
# Import functions specific to this study
sys.path.append(cwd)
from updated_netcoloc_functions import *
from analysis_functions import *
from plotting_functions import *

## Load input gene sets and interaction network


### Note - not needed for the enrichmentanalyses...??

In [4]:
#KT###  upload my NPS-scores:
# with the header: gene, NPS_r, NPS_h, NPS_hr (r=dog)
nps_df = pd.read_csv(DATADIR + "outputs/ocd_ccd_zcomb_z12_251022.txt", sep = "\t", index_col=0)
nps_df.head()

,NPS_r,NPS_h,NPS_hr
gene,,,
TGFBR1,2.074439,0.286812,0.594974
TGFBR2,1.872690,-0.609285,-1.141002
SMAD3,1.040744,0.070344,0.073211
TGFB1,-0.019932,-0.318449,0.006347
SMAD2,1.073654,-0.977044,-1.049007


In [5]:
## Load the final seed gene lists

#I also need the info from seed genes:
# load human seed genes
seed_bin_human_BMI = pd.read_csv('/proj/snic2022-6-229/private/Darwin_Affy/NEW_DA_IMP_MERGED/2024-02-08/netcoloc/NetColoc/data/OCDgenes_v4.txt')
print("Number of human BMI seeds:", len(seed_bin_human_BMI))

#remove genes not found in PCNet:
seed_bin_human_BMI = seed_bin_human_BMI[seed_bin_human_BMI['gene'].isin(nps_df.index)]
print("Number of human BMI seeds:", len(seed_bin_human_BMI))

# load dog seed genes
seed_bin_rat_BMI = pd.read_csv('/proj/snic2022-6-229/private/Darwin_Affy/NEW_DA_IMP_MERGED/2024-02-08/netcoloc/NetColoc/data/CCDgenes_Oct25.txt')
print("Number of rat BMI seeds:", len(seed_bin_rat_BMI))

#remove genes not found in PCNet:
seed_bin_rat_BMI = seed_bin_rat_BMI[seed_bin_rat_BMI['gene'].isin(nps_df.index)]
print("Number of rat BMI seeds:", len(seed_bin_rat_BMI))

Number of human BMI seeds: 48
Number of human BMI seeds: 48
Number of rat BMI seeds: 83
Number of rat BMI seeds: 82


In [6]:
# If you receive a connection error, please retry.
pc_nodes, G_PC = load_pcnet()

number of nodes:
19267

number of edges:
3852119


## Load the conserved network
### OBS No longer needed for the analyses done in this script...???
### needed if loading to cytoscape



Note: the conserved network will remain private until publication. As an alternative to importing the network from NDEx it can be loaded from a stored file (check this)

In [7]:
# KT run this: #OBS the uuid is now for our OCDv4_CCDOct25:# import the conserved network
#ndex_server='public.ndexbio.org'
#ndex_user=getpass("Username:")
#ndex_password=getpass("Password:")
#G_BMI_coloc_cx = ndex2.create_nice_cx_from_server(
#            ndex_server, 
#            username=ndex_user, 
#            password=ndex_password, 
#            uuid='be83bd0e-af34-11f0-a218-005056ae3c32'
#        )

# print out interactome num nodes and edges for diagnostic purposes
#G_BMI_coloc = G_BMI_coloc_cx.to_networkx()
#print('number of nodes:', len(G_BMI_coloc.nodes))
#print('\nnumber of edges:', len(G_BMI_coloc.edges))

number of nodes: 52

number of edges: 64


# Hierarchical Community Detection with HiDeF

**Note:** if you rerun, the communities will not exactly match those in the reported analysis as the hierarchical community detection process is not deterministic. 

In [8]:
#load hierarchy data from netcoloc run:
G_BMI_hier = pd.read_csv("/proj/snic2022-6-229/private/Darwin_Affy/NEW_DA_IMP_MERGED/2024-02-08/netcoloc/NetColoc/example_notebooks/CompulsiveNetwork_hierarchy_data_251022.tsv", sep="\t", index_col=0)
G_BMI_hier.head()

,CD_MemberList,CD_MemberList_Size,CD_Labeled,CD_MemberList_LogSize,CD_CommunityName,CD_AnnotatedMembers,CD_AnnotatedMembers_Size,CD_AnnotatedMembers_Overlap,CD_AnnotatedMembers_Pvalue,HiDeF_persistence,represents,name
0,UNC80 NES STXBP1 CDH8 CNTN1 HCN2 CDH12 ARIH2 O...,165,False,7.366,NaN,NaN,0,0.0,0.0,41,C184,C184
1,UNC80 STXBP1 CDH8 CNTN1 HCN2 CDH12 OPCML RBFOX...,105,False,6.714,NaN,NaN,0,0.0,0.0,94,C185,C185
2,SPMIP2 GALK2 ENC1 NES P4HTM NECAB3 KLHL33 IQCC...,57,False,5.833,NaN,NaN,0,0.0,0.0,102,C186,C186
3,ADGRF3 ADGRB3 GPR137 NECAB3 CACNB2 PDE1B AKAP6...,13,False,3.700,NaN,NaN,0,0.0,0.0,10,C197,C197
4,PDE4D CACNB2 AKAP6 PDE1B,4,False,2.000,NaN,NaN,0,0.0,0.0,65,C224,C224


In [9]:
## Load systems map from netcoloc run
G_BMI_hier_df = pd.read_csv("/proj/snic2022-6-229/private/Darwin_Affy/NEW_DA_IMP_MERGED/2024-02-08/netcoloc/CrossSpeciesBMI/Data/output/OCDv4_CCD_Oct25_systems_map.txt", sep="\t", index_col=0)
G_BMI_hier_df.head(5)

,abnormal synaptic transmission:-log(OR_p),abnormal synaptic transmission:log_OR,abnormal synaptic transmission:num_genes,abnormal synaptic transmission:gene_ids,abnormal synaptic physiology:-log(OR_p),abnormal synaptic physiology:log_OR,abnormal synaptic physiology:num_genes,abnormal synaptic physiology:gene_ids,abnormal CNS synaptic transmission:-log(OR_p),abnormal CNS synaptic transmission:log_OR,...,abnormal miniature inhibitory postsynaptic currents:log_OR,abnormal miniature inhibitory postsynaptic currents:num_genes,abnormal miniature inhibitory postsynaptic currents:gene_ids,gprofiler_name,num_d1_seeds,num_d2_seeds,frac_d1_seeds,frac_d2_seeds,frac_both_seeds,frac_no_seeds
C184,21.660168,1.752352,43,ADGRB3 ARIH1 ATP1A2 BSN CACNA1H CACNG3 CDH8 CN...,21.482295,1.744661,43,ADGRB3 ARIH1 ATP1A2 BSN CACNA1H CACNG3 CDH8 CN...,18.469244,1.681011,...,2.588589,6,CNTNAP4 GABRA1 GABRA2 GABRB3 LRFN3 NLGN1,OCDv4-CCD_Oct25 systems map,6,8,0.036364,0.048485,0.0,0.915152
C185,31.744465,2.392071,42,ADGRB3 ATP1A2 BSN CACNA1H CACNG3 CDH8 CNTNAP4 ...,31.551950,2.384388,42,ADGRB3 ATP1A2 BSN CACNA1H CACNG3 CDH8 CNTNAP4 ...,27.561062,2.281286,...,3.065518,6,CNTNAP4 GABRA1 GABRA2 GABRB3 LRFN3 NLGN1,Protein-protein interactions at synapses,2,4,0.019048,0.038095,0.0,0.942857
C186,0.676815,-1.263990,1,ARIH1,0.682356,-1.271414,1,ARIH1,0.608252,-1.169872,...,0.000000,0,NaN,Antigen processing: Ubiquitination & Proteasom...,4,4,0.070175,0.070175,0.0,0.859649
C197,0.103017,0.278887,1,ADGRB3,0.100004,0.271464,1,ADGRB3,0.142547,0.372992,...,0.000000,0,NaN,C197,0,0,0.000000,0.000000,0.0,1.000000
C224,-0.000000,0.000000,0,NaN,-0.000000,0.000000,0,NaN,-0.000000,0.000000,...,0.000000,0,NaN,C224,0,0,0.000000,0.000000,0.0,1.000000


In [10]:
hier_df_genes = G_BMI_hier.loc[:, ["name","CD_MemberList", "represents"]]
hier_df_genes = hier_df_genes.set_index('name')
hier_df_genes.head()

,CD_MemberList,represents
name,,
C184,UNC80 NES STXBP1 CDH8 CNTN1 HCN2 CDH12 ARIH2 O...,C184
C185,UNC80 STXBP1 CDH8 CNTN1 HCN2 CDH12 OPCML RBFOX...,C185
C186,SPMIP2 GALK2 ENC1 NES P4HTM NECAB3 KLHL33 IQCC...,C186
C197,ADGRF3 ADGRB3 GPR137 NECAB3 CACNB2 PDE1B AKAP6...,C197
C224,PDE4D CACNB2 AKAP6 PDE1B,C224


In [11]:
G_hier_df_genes = G_BMI_hier_df.loc[:, ["frac_d1_seeds", "frac_d2_seeds"]]
G_hier_df_genes.head()

,frac_d1_seeds,frac_d2_seeds
C184,0.036364,0.048485
C185,0.019048,0.038095
C186,0.070175,0.070175
C197,0.000000,0.000000
C224,0.000000,0.000000


In [12]:


# Merge the two DataFrames on their index
merged_df = pd.merge(hier_df_genes, G_hier_df_genes, left_index=True, right_index=True, how='inner')

# merged_df now contains columns: "CD_MemberList", "represents", "frac_d1_seeds", "frac_d2_seeds"
merged_df.head()

,CD_MemberList,represents,frac_d1_seeds,frac_d2_seeds
C184,UNC80 NES STXBP1 CDH8 CNTN1 HCN2 CDH12 ARIH2 O...,C184,0.036364,0.048485
C185,UNC80 STXBP1 CDH8 CNTN1 HCN2 CDH12 OPCML RBFOX...,C185,0.019048,0.038095
C186,SPMIP2 GALK2 ENC1 NES P4HTM NECAB3 KLHL33 IQCC...,C186,0.070175,0.070175
C197,ADGRF3 ADGRB3 GPR137 NECAB3 CACNB2 PDE1B AKAP6...,C197,0.000000,0.000000
C224,PDE4D CACNB2 AKAP6 PDE1B,C224,0.000000,0.000000


In [14]:
# Check columns and first few rows
print(merged_df.head())
print(merged_df.columns)
print(merged_df.index)

# Ensure data types
merged_df['CD_MemberList'] = merged_df['CD_MemberList'].astype(str)
merged_df['represents'] = merged_df['represents'].astype(str)
merged_df['frac_d1_seeds'] = pd.to_numeric(merged_df['frac_d1_seeds'], errors='coerce')
merged_df['frac_d2_seeds'] = pd.to_numeric(merged_df['frac_d2_seeds'], errors='coerce')

# Re-check after conversion
print(merged_df.dtypes)


                                          CD_MemberList represents  \
C184  UNC80 NES STXBP1 CDH8 CNTN1 HCN2 CDH12 ARIH2 O...       C184   
C185  UNC80 STXBP1 CDH8 CNTN1 HCN2 CDH12 OPCML RBFOX...       C185   
C186  SPMIP2 GALK2 ENC1 NES P4HTM NECAB3 KLHL33 IQCC...       C186   
C197  ADGRF3 ADGRB3 GPR137 NECAB3 CACNB2 PDE1B AKAP6...       C197   
C224                           PDE4D CACNB2 AKAP6 PDE1B       C224   

      frac_d1_seeds  frac_d2_seeds  
C184       0.036364       0.048485  
C185       0.019048       0.038095  
C186       0.070175       0.070175  
C197       0.000000       0.000000  
C224       0.000000       0.000000  
Index(['CD_MemberList', 'represents', 'frac_d1_seeds', 'frac_d2_seeds'], dtype='object')
Index(['C184', 'C185', 'C186', 'C197', 'C224', 'C194', 'C227', 'C199', 'C208',
       'C212', 'C214', 'C187', 'C222', 'C196', 'C188', 'C189', 'C209', 'C205',
       'C190', 'C211', 'C206', 'C198', 'C191', 'C193', 'C192', 'C232', 'C226',
       'C233', 'C200', 'C202', 

# Community Annotation

## Seed gene fractions

## Initial Annotation with GO

Annotate with moderately sized systems (between 50 to 1000 genes per system) if the community shows a significant ($p<1\times 10^{-4}$) enrichment for the term. Also require that the GO term and community and share at least 3 genes to increase the stringency of the annotation. Label the system using the GO term that meets these criteria, and has the highest sum of precision and recall. Systems which have no GO terms meeting these criteria are labeled with their unique system ID.

**Note:**  
Rerunning the annotation may generate slightly different results due to changes in the GO:BP database over time. The original results were generated January 18, 2022.

In [12]:
def filter_go_annotations(go_df, term_min=50, term_max=1000, p_th=1e-4, min_intersection=3):
    """Filters available annotations for a community based on specificity and significance.
    Args:
        go_df (pandas.DataFrame): All available significant GO terms for each community
        term_min (int, optional):   The minimum size of a term to keep. Defaults to 50.
        term_max (int, optional): The maximum size of a term to keep. Defaults to 1000.
        p_th (float, optional): The significance threshold. Defaults to 1e-4.
        min_intersection (int, optional): Minimum number of community terms annotated to the GO term. Defaults to 3.

    Returns:
        pandas.DataFrame: A filter dataframe of GO annotations per community, sorted by sum of precision and recall.
    """
    go_df = go_df[(go_df['term_size'] <= term_max) & (go_df['term_size'] >= term_min)]
    go_df = go_df[go_df['intersection_size'] >= min_intersection]
    go_df = go_df[go_df['p_value'] < p_th] # set a stringent pvalue threshold
    go_df['sum_PR'] = go_df['recall'] + go_df['precision']
    go_df = go_df.sort_values('sum_PR',ascending=False)
    return go_df

In [30]:
## Obs load created file instead of rerun
# Annotate with GO terms gprofiler annotation of clusters
system_names = {}
all_results = []
BMI_hier_df = merged_df
for comm in BMI_hier_df.index.tolist():
    # get the genes in the community
    if isinstance(BMI_hier_df['CD_MemberList'].loc[comm], str):
        focal_genes=BMI_hier_df['CD_MemberList'].loc[comm].split(' ')
    else:
        focal_genes=BMI_hier_df['CD_MemberList'].loc[comm]
    print(comm,":", len(focal_genes))
    system_names[comm] = comm # default is to keep identifier as name, will be replaced if confident annotation available
    if len(focal_genes)>2:
        # get all available GO:BP annotations
        gp_temp = pd.DataFrame(gp.profile(focal_genes,significance_threshold_method='fdr',
                                            sources=['GO:BP'], no_evidences=False))
        if len(gp_temp)>0: # make sure data is not empty
            # filter and append the results
            gp_temp["community"] = comm
            all_results.append(gp_temp)
            gp_temp = filter_go_annotations(gp_temp, 50, 1000, 1e-3, 3)
            if len(gp_temp)>1:
                system_names[comm] = gp_temp.head(1)['name'].tolist()[0]
                #all_results.append(gp_temp)
go_results = pd.concat(all_results)
#go_results.to_csv(DATADIR+"outputs/Compulsive_hierachy_full_GO_enrichment_251023.tsv", index=True, sep="\t")

C184 : 165
C185 : 105
C186 : 57
C197 : 13
C224 : 4
C194 : 18
C227 : 4
C199 : 11
C208 : 7
C212 : 7
C214 : 6
C187 : 44
C222 : 5
C196 : 14
C188 : 39
C189 : 37
C209 : 7
C205 : 8
C190 : 32
C211 : 7
C206 : 7
C198 : 11
C191 : 25
C193 : 18
C192 : 22
C232 : 4
C226 : 4
C233 : 4
C200 : 11
C202 : 9
C195 : 18
C210 : 7
C204 : 8
C201 : 10
C221 : 5
C223 : 5
C215 : 6
C219 : 5
C228 : 4
C207 : 7
C203 : 9
C216 : 6
C218 : 6
C234 : 4
C217 : 6
C220 : 5
C213 : 6
C225 : 4
C235 : 4
C230 : 4
C231 : 4
C229 : 4


In [13]:
#read the one run instead of rerunning:
go_results = pd.read_csv(DATADIR+"outputs/Compulsive_hierachy_full_GO_enrichment_251023.tsv",  sep="\t", index_col=0)
go_results.head(5)

,description,effective_domain_size,intersection_size,intersections,name,native,p_value,parents,precision,query,query_size,recall,significant,source,term_size,evidences,community
0,"""Cell-cell signaling in either direction acros...",21026,43,"['STXBP1', 'CDH8', 'CACNB2', 'GRM7', 'DLGAP1',...",trans-synaptic signaling,GO:0099537,2.871846e-23,['GO:0099536'],0.282895,query_1,152,0.055058,True,GO:BP,781,"[['ISS', 'IBA', 'TAS', 'IEA'], ['IEA'], ['IBA'...",C184
1,"""Cell-cell signaling from pre to post-synapse,...",21026,43,"['STXBP1', 'CDH8', 'CACNB2', 'GRM7', 'DLGAP1',...",anterograde trans-synaptic signaling,GO:0098916,2.871846e-23,['GO:0099537'],0.282895,query_1,152,0.055484,True,GO:BP,775,"[['ISS', 'IBA', 'TAS', 'IEA'], ['IEA'], ['IBA'...",C184
2,"""The vesicular release of classical neurotrans...",21026,43,"['STXBP1', 'CDH8', 'CACNB2', 'GRM7', 'DLGAP1',...",chemical synaptic transmission,GO:0007268,2.871846e-23,['GO:0098916'],0.282895,query_1,152,0.055484,True,GO:BP,775,"[['ISS', 'IBA', 'TAS', 'IEA'], ['IEA'], ['IBA'...",C184
3,"""Any process that modulates the establishment ...",21026,35,"['HCN2', 'CACNB2', 'ATP1A2', 'GABRB3', 'KCNB1'...",regulation of membrane potential,GO:0042391,3.685135e-23,['GO:0065008'],0.230263,query_1,152,0.076419,True,GO:BP,458,"[['IMP', 'IBA', 'IC', 'IEA'], ['IMP'], ['TAS',...",C184
4,"""Cell-cell signaling to, from or within a syna...",21026,43,"['STXBP1', 'CDH8', 'CACNB2', 'GRM7', 'DLGAP1',...",synaptic signaling,GO:0099536,6.386167e-23,['GO:0007267'],0.282895,query_1,152,0.053284,True,GO:BP,807,"[['ISS', 'IBA', 'TAS', 'IEA'], ['IEA'], ['IBA'...",C184


# Validation


## Load Mouse Genome Database (MGD) data

**Note:** Developers uses a stable version of MGD downloaded on April 24, 2022. If using an updated dataset, slight variations in results are expected due to recent data submission to MGD. 

We used a download from: Jan 30th 2025

In [16]:
# load mappings between genes and mammalian phenotypes
mgi_df = load_MGI_mouseKO_data(url='http://www.informatics.jax.org/downloads/reports/MGI_PhenoGenoMP.rpt', 
                                map_using="mgi", update=False, data_loc=DATADIR+"Reference/") #changed update to True when I have downloaded it once at Jan 30th 2025
print("Number of gene-phenotype associations:", len(mgi_df))

Number of gene-phenotype associations: 399047


**Node Updating**  
The gene symbols listed under human orthologs in the MGD are more up to date than the gene symbols in PCNet. Therefore, to ensure correct mapping of our conserved network to MGD we map the human ortholog column to the symbols in PCNet using MyGeneInfo. Some gene symbols return duplicate entries via MyGeneInfo which we resolved manually (see `symbol_list` below).

In [17]:
symbol_list = {'GARS': 'GARS1', 'GGT2': 'GGT2P','MARS': 'MARS1', 'HIST1H2BC': 'H2BC4', 'GBA': 'GBA1', 'HIST1H2BG': 'H2BC8',
'HIST1H2BF': 'H2BC7', 'HIST1H2BE': 'H2BC6', 'HIST1H2BI': 'H2BC10', 'CBWD3': 'ZNG1C', 'BHLHB9': 'GPRASP3', 'ELOA3': 'ELOA3P',
'ELOA3B': 'ELOA3BP', 'SMIM11B': 'SMIM11', 'CBSL': 'CBS', 'C2orf27B': 'CDRT15P3', 'LINC00846': 'TCP10L'}
updated_pc_nodes, _ = update_nodes(pc_nodes, symbol_list)

167 input query terms found dup hits:	[('CAST', 2), ('DUX4L9', 4), ('IGHV1-69', 2), ('IGHV3-48', 2), ('IGHV3-33', 2), ('IGHV3-30', 2), ('I
91 input query terms found no hit:	['MFSD10', 'ILVBL', 'ND5', 'COX2', 'NDUFA4', 'BMT2', 'FAM210A', 'NEPRO', 'ARMT1', 'NDUFA4L2', 'TMEM1
5 input query terms found dup hits:	[('COX2', 2), ('NDUFA4', 2), ('C18orf21', 2), ('C17orf49', 2), ('CIR1', 2)]
10 input query terms found no hit:	['ND5', 'ND2', 'ND4', 'ND3', 'ND4L', 'ND6', 'CYTB', 'ATP8', 'ATP6', 'TEC_100124696']


In [18]:
# updated the genotype-phenotype identifiers
mgi_df = change_symbols(mgi_df, updated_pc_nodes)

In [19]:
# load the mammalian phenotype ontology
MPO = load_MPO(url='http://www.informatics.jax.org/downloads/reports/MPheno_OBO.ontology', 
                        use_genes=True, mapping=mgi_df, data_loc=DATADIR+"Reference/", update=False) #changed update to True when I have downloaded it once at Jan 30th 2025

15418


In [20]:
# summarize the total evidence per gene and term
term_counts, gene_mapping, term_mapping = genes_per_node(MPO)

In [21]:
# construct a graph object of the MPO
mp_graph = get_mp_graph(DATADIR+"Reference/parsed_mp.txt")

## Perform enrichment of Mammalian Phenotypes

### All phenotypes in MGD:

In [25]:
### Obs this is already done for all phenotypes, read in the results below:
#bmi_phenotypes = []
#G_BMI_hier_df = merged_df
########## for each community, calculate the enrichment for all phenotypes##############
#for community in tqdm(G_BMI_hier_df.index):
#    x = community_term_enrichment(community,  G_BMI_hier_df, MPO, mgi_df, term_counts, gene_mapping, G_int=G_PC)
#    x = x.assign(name=community)
#    bmi_phenotypes.append(x)
#bmi_pheno_results = pd.concat(bmi_phenotypes)
#bmi_pheno_results.to_csv(DATADIR + "outputs/251023_KT_hierarchy_full_MGD_enrichment_results.tsv", sep="\t") 

bmi_pheno_results = pd.read_csv(DATADIR+"outputs/251023_KT_hierarchy_full_MGD_enrichment_results.tsv", sep="\t", index_col=0)
bmi_pheno_results.head()
    

,observed,total,OR,OR_p,OR_CI_lower,OR_CI_upper,hyper_p,sig_5e6,description,size,name
MP:0000001,143,14058,2.422961,1.159039e-04,1.544962,3.799924,6.521084e-06,False,mammalian phenotype,165,C184
MP:0004924,98,5758,3.473751,5.440006e-15,2.542094,4.746855,8.525396e-16,True,abnormal behavior,165,C184
MP:0005386,98,5760,3.472008,5.577483e-15,2.540818,4.744470,8.741826e-16,True,behavior/neurological phenotype,165,C184
MP:0003631,83,4602,3.266396,4.118873e-14,2.402949,4.440103,4.381803e-14,True,nervous system phenotype,165,C184
MP:0005378,79,6707,1.728828,4.718188e-04,1.271953,2.349810,1.942339e-04,False,growth/size/body region phenotype,165,C184


In [26]:
# assign the community annotations
bmi_pheno_results = bmi_pheno_results.merge(final_annotations, left_on="name", right_index=True)
bmi_pheno_results.value_counts("description")

description
mammalian phenotype                                   52
abnormal behavior                                     52
behavior/neurological phenotype                       52
growth/size/body region phenotype                     51
abnormal motor capabilities/coordination/movement     50
                                                      ..
decreased fluid intake                                 2
small L4 dorsal root ganglion                          2
decreased dopaminergic neuron number                   2
increased abdominal fat pad weight                     2
abnormal cochlear OHC afferent innervation pattern     2
Length: 2353, dtype: int64

## Identify communities enriched for any of the MPO terms!! i.e. check them all:

In [44]:
# Identify the communities with sign enrichment:
comm_results_sign = bmi_pheno_results.sort_values(by="OR_CI_lower", ascending=False)
comm_results_sign["q"] = fdrcorrection(list(comm_results_sign.OR_p.values), method="poscorr")[1]
comm_results_sign["-log10p"] = comm_results_sign.OR_p.apply(lambda x: -1 * np.log10(x))
comm_results_sign["annotate"] = comm_results_sign.apply(lambda x: x["name"] if (x["OR_p"] < 0.05) else "Other", axis=1)
#comm_results_sign.to_csv(DATADIR + "outputs/SIGN_251023_KT_hierarchy_full_MGD_enrichment_results.tsv", sep="\t") 

In [27]:
#read in sign results MGD
comm_results_sign = pd.read_csv(DATADIR+"outputs/SIGN_251023_KT_hierarchy_full_MGD_enrichment_results.tsv", sep="\t", index_col=0)

In [28]:
comm_results_sign.head()

,observed,total,OR,OR_p,OR_CI_lower,OR_CI_upper,hyper_p,sig_5e6,description,size,name,annotation,q,-log10p,annotate
MP:0010859,2,4,3209.500000,7.733007e-14,386.379585,26660.027233,1.879057e-10,True,abnormal anterior commissure pars anterior mor...,8,C204,axon guidance,2.125948e-11,13.111652,C204
MP:0012718,1,1,12842.000000,2.199010e-07,358.547152,459958.929327,0.000000e+00,True,decreased hemangioblast number,4,C224,C224,8.318592e-06,6.657773,C224
MP:0012715,1,2,6420.666667,9.497534e-09,321.613170,128181.816858,3.232776e-08,True,abnormal hemangioblast morphology,4,C224,C224,7.174128e-07,8.022389,C224
MP:0011247,1,2,6420.666667,9.497534e-09,321.613170,128181.816858,3.232776e-08,True,abnormal yolk sac hematopoietic stem cell morp...,4,C224,C224,7.174128e-07,8.022389,C224
MP:0012716,1,2,6420.666667,9.497534e-09,321.613170,128181.816858,3.232776e-08,True,abnormal hemangioblast number,4,C224,C224,7.174128e-07,8.022389,C224


In [29]:
comm_results_sign.value_counts("description")


description
mammalian phenotype                                   52
abnormal behavior                                     52
behavior/neurological phenotype                       52
growth/size/body region phenotype                     51
abnormal motor capabilities/coordination/movement     50
                                                      ..
decreased fluid intake                                 2
small L4 dorsal root ganglion                          2
decreased dopaminergic neuron number                   2
increased abdominal fat pad weight                     2
abnormal cochlear OHC afferent innervation pattern     2
Length: 2353, dtype: int64